In [1]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for flasher scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_V13 = 1    # 8B
POINTS_V14 = 5.5833   # 8B+
POINTS_V15 = 33.5   # 8C
POINTS_V16 = 1    # 8C+

TOP_LABELS = 12               # label top N names
JITTER = 0.025
LABELED_X_SPREAD = 1.15

HIGHLIGHT_NAME = "Adam Ondra" # set to None if you don't want special highlight
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/20/hard-boulder-problem-flashes"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

font_to_v = {"8B": 13, "8B+": 14, "8C": 15, "8C+": 16}
score_map = {13: POINTS_V13, 14: POINTS_V14, 15: POINTS_V15, 16: POINTS_V16}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade", "climb"}]
df = df[keep].copy()
# Remove Adam Ondra's two recent 8C flashes
df = df[~(
    (df["Climber"] == "Adam Ondra") &
    (df["Climb"].str.contains("Lion", case=False, na=False) |
     df["Climb"].str.contains("Celestite", case=False, na=False))
)]

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["V"] = df["Grade_clean"].map(font_to_v)
df["Score"] = df["V"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions (x is meaningless) ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))
X_OFFSET = -0.074
rank["x"] = rank["x"] + X_OFFSET




# --- Choose which names to label (top N, plus highlight name if present) ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Improve label spacing ---
# Alternate labeled names left/right so nearby labels don't stack
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points farther horizontally for extra separation
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left-side vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Boulder Flashers (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.12, 0.12],
    ),
    yaxis=dict(
        title="Flash Score",
        tickfont=dict(size=20),
        showgrid=False   # removes horizontal gridlines
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)


fig.show()

out_html = "flashers_vertical_strip_outlier.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: flashers_vertical_strip_outlier.html


In [3]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for flasher scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_V13 = 1    # 8B
POINTS_V14 = 5    # 8B+
POINTS_V15 = 18   # 8C
POINTS_V16 = 1    # 8C+

TOP_LABELS = 12
JITTER = 0.025
LABELED_X_SPREAD = 1.15
HIGHLIGHT_NAME = "Adam Ondra"
RANDOM_SEED = 7

UNLABELED_X_OFFSET = -0.105
LABELED_X_OFFSET = -0.09

HIGHLIGHT_X = -0.105

# NEW SIZE CONTROLS
NORMAL_DOT_SIZE = 16
ONDRA_DOT_SIZE = 25
# ============================

URL = "https://climbing-history.org/list/20/hard-boulder-problem-flashes"


def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s


font_to_v = {"8B": 13, "8B+": 14, "8C": 15, "8C+": 16}
score_map = {13: POINTS_V13, 14: POINTS_V14, 15: POINTS_V15, 16: POINTS_V16}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade", "climb"}]
df = df[keep].copy()

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["V"] = df["Grade_clean"].map(font_to_v)
df["Score"] = df["V"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"Score": "TotalScore"})
)

# ============================
# ADD 18 POINTS TO ONDRA
# ============================
rank.loc[rank["Climber"] == "Adam Ondra", "TotalScore"] += 18

# --- Base jitter x positions ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))

# --- Choose labels ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

label_mask = rank["label"] != ""
unlabeled_mask = ~label_mask

rank.loc[unlabeled_mask, "x"] = rank.loc[unlabeled_mask, "x"] + UNLABELED_X_OFFSET
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] + LABELED_X_OFFSET

# --- Label spacing ---
rank["textpos"] = "middle right"
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# --- Force Ondra farther left ---
if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "x"] = HIGHLIGHT_X

# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below",
    )
)

# OTHER CLIMBERS
fig.add_trace(
    go.Scatter(
        x=others["x"],
        y=others["TotalScore"],
        mode="markers+text",
        text=others["label"],
        textposition=others["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([others["Climber"]], axis=-1),
        marker=dict(size=NORMAL_DOT_SIZE, opacity=0.9),
        name="Climbers",
    )
)

# ONDRA
if len(hilite):
    fig.add_trace(
        go.Scatter(
            x=hilite["x"],
            y=hilite["TotalScore"],
            mode="markers+text",
            text=hilite["label"],
            textposition=hilite["textpos"],
            hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
            customdata=np.stack([hilite["Climber"]], axis=-1),
            marker=dict(
                size=ONDRA_DOT_SIZE,
                opacity=1.0,
                symbol="diamond"
            ),
            name=HIGHLIGHT_NAME,
        )
    )

# axis line
fig.add_shape(
    type="line",
    x0=0.02,
    x1=0.02,
    y0=0,
    y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Boulder Flashers (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.14, 0.12],
    ),
    yaxis=dict(
        title="Flash Score",
        tickfont=dict(size=50),
        showgrid=False,
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "flashers_vertical_strip_outlier.html"
fig.write_html(out_html)

print("Saved:", out_html)

Saved: flashers_vertical_strip_outlier.html


In [4]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything)
# for hardest boulder ascent scores (8C+ / 9A / 9A+)

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_V16 = 1    # 8C+
POINTS_V17 = 5    # 9A
POINTS_V18 = 18   # 9A+

TOP_LABELS = 12
JITTER = 0.10
LABELED_X_SPREAD = 1.8
HIGHLIGHT_NAME = "Will Bosi"
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/14/hard-boulder-problem-ascents"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

font_to_v = {
    "8C+": 16,
    "9A": 17,
    "9A+": 18,
}

score_map = {
    16: POINTS_V16,
    17: POINTS_V17,
    18: POINTS_V18,
}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade", "climb"}]
df = df[keep].copy()

# Remove Adam Ondra's Lion's Share and Celestite flashes
df = df[~(
    (df["Climber"] == "Adam Ondra") &
    (df["Climb"].str.contains("Share", case=False, na=False) |
     df["Climb"].str.contains("Celestite", case=False, na=False))
)]

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["V"] = df["Grade_clean"].map(font_to_v)
df["Score"] = df["V"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))

# --- Label top climbers ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Alternate label sides ---
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# --- Manual fixes for overlapping labels ---
rank.loc[rank["Climber"] == "Hamish McArthur", "textpos"] = "middle right"
rank.loc[rank["Climber"] == "Noah Wheeler", "textpos"] = "middle left"

# --- Spread labeled points ---
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# split highlight vs others
is_highlight = rank["Climber"] == HIGHLIGHT_NAME
others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Hardest Boulder Ascensionists (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.35, 0.35],
    ),
    yaxis=dict(title="Ascent Score"),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "hardest_boulder_ascents_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: hardest_boulder_ascents_vertical_strip.html


In [4]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything)
# for hard sport climbing ascent scores (9b / 9b+ / 9c)

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_9B = 1
POINTS_9B_PLUS = 5
POINTS_9C = 18

TOP_LABELS = 12
JITTER = 0.10
LABELED_X_SPREAD = 1.8
HIGHLIGHT_NAME = "Adam Ondra"
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/5/hard-sport-climbing-ascents"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

score_map = {
    "9B": POINTS_9B,
    "9B+": POINTS_9B_PLUS,
    "9C": POINTS_9C,
}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade"}]
df = df[keep].copy()

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["Score"] = df["Grade_clean"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))

# --- Label top climbers ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# Always label Will Bosi too
rank.loc[rank["Climber"] == "Will Bosi", "label"] = "Will Bosi"

# --- Alternate label sides ---
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# --- Spread labeled points horizontally ---
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# --- Manual overlap fixes for crowded lower cluster ---
# Put each crowded name on a deliberate side
# --- Manual overlap fixes for crowded lower cluster ---

rank.loc[rank["Climber"] == "Yannick Flohé", "textpos"] = "middle left"
rank.loc[rank["Climber"] == "Jonatan Flor", "textpos"] = "top center"
rank.loc[rank["Climber"] == "Eduardo Zorzi", "textpos"] = "middle right"

# Put Will Bosi to the right with label above
rank.loc[rank["Climber"] == "Will Bosi", "textpos"] = "top center"

# x adjustments
rank.loc[rank["Climber"] == "Yannick Flohé", "x"] -= 0.02
rank.loc[rank["Climber"] == "Jonatan Flor", "x"] -= 0.01
rank.loc[rank["Climber"] == "Eduardo Zorzi", "x"] += 0.02

# push Will Bosi further right
rank.loc[rank["Climber"] == "Will Bosi", "x"] += 0.1
# split highlight vs others
is_highlight = rank["Climber"] == HIGHLIGHT_NAME if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)
others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Hardest Sport Ascensionists (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.35, 0.35],
    ),
    yaxis=dict(title="Sport Score"),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "hardest_sport_ascents_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: hardest_sport_ascents_vertical_strip.html


In [ ]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything)
# for hard sport climbing onsight scores (8c+ / 9a)

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_8C_PLUS = 1
POINTS_9A = 6

TOP_LABELS = 12
JITTER = 0.10
LABELED_X_SPREAD = 1.8
HIGHLIGHT_NAME = "Adam Ondra"
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/37/hard-sport-climbing-onsights"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

score_map = {
    "8C+": POINTS_8C_PLUS,
    "9A": POINTS_9A,
}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade"}]
df = df[keep].copy()

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["Score"] = df["Grade_clean"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))

# --- Label top climbers ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Alternate label sides ---
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# --- Spread labeled points horizontally ---
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# ============================
# MANUAL LABEL FIXES
# ============================

# Domen Škofic → label above
mask_domen = rank["Climber"].str.contains("Domen", case=False, na=False)
rank.loc[mask_domen, "textpos"] = "top center"
rank.loc[mask_domen, "x"] += 0.01

# Edu Marín Puigblanque → label left
mask_puig = rank["Climber"].str.contains("Puigblanque", case=False, na=False)
rank.loc[mask_puig, "textpos"] = "middle left"
rank.loc[mask_puig, "x"] -= 0.02

# Laura Rogora → label above
mask_rogora = rank["Climber"].str.contains("Rogora", case=False, na=False)
rank.loc[mask_rogora, "textpos"] = "top center"

# Patxi Usobiaga → label right
mask_patxi = rank["Climber"].str.contains("Usobiaga", case=False, na=False)
rank.loc[mask_patxi, "textpos"] = "middle right"
rank.loc[mask_patxi, "x"] += 0.02

# ============================

# split highlight vs others
is_highlight = rank["Climber"] == HIGHLIGHT_NAME if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)
others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Hardest Onsighters (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.35, 0.35],
    ),
    yaxis=dict(title="Onsight Score"),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "hardest_sport_onsights_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

In [ ]:
import pandas as pd
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/Ranking_of_career_IFSC_victories_by_climber"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# Fetch page manually to avoid 403
html = requests.get(url, headers=headers, timeout=30).text
tables = pd.read_html(StringIO(html))

# Men's table is table 1 from your debug output
mens = tables[1].copy()

# Flatten columns using the LAST level of the MultiIndex
if isinstance(mens.columns, pd.MultiIndex):
    mens.columns = [col[-1] for col in mens.columns]
else:
    mens.columns = [str(c).strip() for c in mens.columns]

print("Columns:", mens.columns.tolist())

# Build climber name
mens["Climber"] = (
    mens["First Name"].astype(str).str.strip() + " " +
    mens["Last Name"].astype(str).str.strip()
)

# Convert numeric columns
for col in ["L", "B", "S", "C", "T", "L1", "B1", "S1", "C1", "T1", "L2", "B2", "S2", "C2", "T2"]:
    if col in mens.columns:
        mens[col] = pd.to_numeric(mens[col], errors="coerce").fillna(0)

# Remove speed from total
mens["T_no_speed"] = mens["L"] + mens["B"] + mens["C"]

# Optional: drop pure speed specialists
mens_nospeed = mens[mens["T_no_speed"] > 0].copy()

# Sort by non-speed total
mens_nospeed = mens_nospeed.sort_values(
    ["T_no_speed", "L", "B", "C"],
    ascending=False
).reset_index(drop=True)

print("\nMEN'S NON-SPEED IFSC VICTORIES")
print(mens_nospeed[["Climber", "L", "B", "S", "C", "T_no_speed"]].head(30))

In [1]:
import pandas as pd
import requests
from io import StringIO
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
TOP_LABELS = 12
JITTER = 0.10
LABELED_X_SPREAD = 1.8
HIGHLIGHT_NAME = "Jakob Schubert"
RANDOM_SEED = 7
# ============================

url = "https://en.wikipedia.org/wiki/Ranking_of_career_IFSC_victories_by_climber"
headers = {"User-Agent": "Mozilla/5.0"}

# --- Load Wikipedia tables ---
html = requests.get(url, headers=headers, timeout=30).text
tables = pd.read_html(StringIO(html))

# Men's table = table 1
mens = tables[1].copy()

# Flatten multi-index columns to the last level
if isinstance(mens.columns, pd.MultiIndex):
    mens.columns = [col[-1] for col in mens.columns]
else:
    mens.columns = [str(c).strip() for c in mens.columns]

# Build climber name
mens["Climber"] = (
    mens["First Name"].astype(str).str.strip() + " " +
    mens["Last Name"].astype(str).str.strip()
)

# Remove repeated header / junk rows that got parsed as data
bad_names = {
    "",
    "nan nan",
    "first name last name",
    "total total",
}
mens = mens[~mens["Climber"].str.strip().str.lower().isin(bad_names)].copy()

# Convert numeric columns
for col in ["L", "B", "S", "C"]:
    if col in mens.columns:
        mens[col] = pd.to_numeric(mens[col], errors="coerce")

# Drop rows where all competition columns are missing
mens = mens.dropna(subset=["L", "B", "S", "C"], how="all").copy()

# Fill remaining NaNs with 0
for col in ["L", "B", "S", "C"]:
    mens[col] = mens[col].fillna(0)

# Compute non-speed competition wins
mens["Score"] = mens["L"] + mens["B"] + mens["C"]

# Remove pure speed specialists
rank = mens[mens["Score"] > 0].copy()

rank = rank[["Climber", "L", "B", "C", "Score"]]

# Sort
rank = rank.sort_values(
    ["Score", "L", "B", "C"],
    ascending=False
).reset_index(drop=True)

# --- Jitter x positions ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))

# --- Label top climbers ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Alternate label sides ---
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# ============================
# MANUAL LABEL FIXES
# ============================

# Petit → label right
mask_petit = rank["Climber"].str.contains("Petit", case=False, na=False)
rank.loc[mask_petit, "textpos"] = "middle right"
rank.loc[mask_petit, "x"] += 0.03

# Chabot → label right
mask_chabot = rank["Climber"].str.contains("Chabot", case=False, na=False)
rank.loc[mask_chabot, "textpos"] = "middle right"
rank.loc[mask_chabot, "x"] += 0.03

# split highlight vs others
is_highlight = rank["Climber"] == HIGHLIGHT_NAME
others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["Score"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Competition wins: %{y}<br>"
        "Lead: %{customdata[1]}<br>"
        "Boulder: %{customdata[2]}<br>"
        "Combined: %{customdata[3]}<extra></extra>"
    ),
    customdata=np.stack(
        [others["Climber"], others["L"], others["B"], others["C"]],
        axis=-1
    ),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["Score"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Competition wins: %{y}<br>"
            "Lead: %{customdata[1]}<br>"
            "Boulder: %{customdata[2]}<br>"
            "Combined: %{customdata[3]}<extra></extra>"
        ),
        customdata=np.stack(
            [hilite["Climber"], hilite["L"], hilite["B"], hilite["C"]],
            axis=-1
        ),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Competition Climbers (Men, No Speed)",
    template="plotly_dark",
    xaxis=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.35, 0.35],
    ),
    yaxis=dict(title="IFSC Wins (Lead + Boulder + Combined)"),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "mens_ifsc_no_speed_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: mens_ifsc_no_speed_vertical_strip.html


In [2]:
import pandas as pd
import requests
from io import StringIO
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
TOP_LABELS = 12
JITTER = 0.10
LABELED_X_SPREAD = 1.8
HIGHLIGHT_NAME = "Janja Garnbret"
RANDOM_SEED = 7
# ============================

url = "https://en.wikipedia.org/wiki/Ranking_of_career_IFSC_victories_by_climber"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(url, headers=headers).text
tables = pd.read_html(StringIO(html))

# women's table
womens = tables[2].copy()

# flatten headers
if isinstance(womens.columns, pd.MultiIndex):
    womens.columns = [c[-1] for c in womens.columns]

# climber name
womens["Climber"] = (
    womens["First Name"].astype(str).str.strip() +
    " " +
    womens["Last Name"].astype(str).str.strip()
)

# remove junk rows
bad = {"", "nan nan", "first name last name", "total total"}
womens = womens[~womens["Climber"].str.lower().isin(bad)]

# numeric columns
for c in ["L", "B", "S", "C"]:
    womens[c] = pd.to_numeric(womens[c], errors="coerce").fillna(0)

# score without speed
womens["Score"] = womens["L"] + womens["B"] + womens["C"]

rank = womens[womens["Score"] > 0].copy()
rank = rank[["Climber","L","B","C","Score"]]

rank = rank.sort_values(
    ["Score","L","B","C"],
    ascending=False
).reset_index(drop=True)

# jitter
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, len(rank))

# labels
rank["label"] = ""
rank.loc[rank.head(TOP_LABELS).index,"label"] = rank["Climber"]

rank.loc[rank["Climber"] == HIGHLIGHT_NAME,"label"] = HIGHLIGHT_NAME

# alternate sides
rank["textpos"] = "middle right"
lab = rank["label"] != ""
idx = rank.index[lab]

for i,iidx in enumerate(idx):
    rank.loc[iidx,"textpos"] = "middle right" if i % 2 == 0 else "middle left"

# spread labels
rank.loc[lab,"x"] *= LABELED_X_SPREAD


# ============================
# MANUAL LABEL FIXES
# ============================

# Grossman → left
m = rank["Climber"].str.contains("Grossman",case=False,na=False)
rank.loc[m,"textpos"] = "middle left"
rank.loc[m,"x"] -= 0.03

# Anna → left
m = rank["Climber"].str.contains("Anna",case=False,na=False)
rank.loc[m,"textpos"] = "middle left"
rank.loc[m,"x"] -= 0.03

# Mina / Miho → top center
m = rank["Climber"].str.contains("Miho|Mina",case=False,na=False)
rank.loc[m,"textpos"] = "top center"

# Sarkany → right
m = rank["Climber"].str.contains("Sark",case=False,na=False)
rank.loc[m,"textpos"] = "middle right"
rank.loc[m,"x"] += 0.03


# split highlight
is_high = rank["Climber"] == HIGHLIGHT_NAME
others = rank[~is_high]
hilite = rank[is_high]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["Score"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    customdata=np.stack(
        [others["Climber"],others["L"],others["B"],others["C"]],
        axis=-1
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Wins: %{y}<br>"
        "Lead %{customdata[1]}<br>"
        "Boulder %{customdata[2]}<br>"
        "Combined %{customdata[3]}<extra></extra>"
    ),
    marker=dict(size=10,opacity=0.85),
    name="Climbers"
))

fig.add_trace(go.Scatter(
    x=hilite["x"],
    y=hilite["Score"],
    mode="markers+text",
    text=hilite["label"],
    textposition=hilite["textpos"],
    customdata=np.stack(
        [hilite["Climber"],hilite["L"],hilite["B"],hilite["C"]],
        axis=-1
    ),
    marker=dict(size=14,symbol="diamond"),
    name=HIGHLIGHT_NAME
))

fig.add_shape(
    type="line",
    x0=0.02,x1=0.02,
    y0=0,y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3,color="white")
)

fig.update_layout(
    title="Best Competition Climbers (Women, No Speed)",
    template="plotly_dark",
    xaxis=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.35,0.35]
    ),
    yaxis=dict(title="IFSC Wins (Lead + Boulder + Combined)"),
    margin=dict(l=80,r=40,t=70,b=60)
)

fig.show()